## Breitensuche (bfs, breadth first search)

[Video Labyrinth](https://youtu.be/J-E54M3VsEc) - [Video MOO](https://youtu.be/xRZvhyEfvr4) -
[Video Girlande](https://youtu.be/7LW-xa43GZc) - [Video 8-Puzzle](https://youtu.be/l5STsdlJ7X4)

Viele Rätsel und Spiele haben dieselbe Form: Wir beginnen in einer Ausgangssituation, dürfen bestimmte Züge machen und
wollen mit möglichst wenigen Zügen ein Ziel erreichen. Beispiele sind der Weg durch ein Labyrinth, das Schiebepuzzle
oder Überfahrten über einen Fluss. Solche **Suchprobleme** lösen wir alle mit demselben Verfahren, der Breitensuche.

Dafür brauchen wir ein paar Begriffe:

- **Zustand** (state): eine Situation im Spiel, z.B. die Position im Labyrinth.
- **Startzustand**: die Situation am Anfang.
- **Zieltest** (goaltest): prüft, ob ein Zustand eine Lösung ist.
- **Folgezustände** (nextstates): alle Zustände, die man von einem Zustand aus mit einem Zug erreicht.

Die Zustände bilden zusammen mit den Zügen einen Graphen: Die Zustände sind die Knoten, ein Zug ist eine Kante. Die
Breitensuche durchsucht diesen Graphen. Sie berechnet ihn aber nicht vorher ganz, sondern erzeugt die Folgezustände
erst, wenn sie einen Zustand untersucht.

#### Beispiel: Labyrinth

Wir wollen in einem Labyrinth (maze) den kürzesten Weg vom Start S zum Ziel Z finden. Ein Zug ist ein Schritt nach
oben, unten, links oder rechts, `#` ist eine Wand. Ein Zustand ist eine Position $(x, y)$: $x$ ist die Zeile, $y$ die
Spalte, $(0, 0)$ ist oben links.

<img src='maze1.png' width='800'>

Rechts im Bild sieht man die Zustände, die die Breitensuche nacheinander untersucht. Die Pfeile zeigen jeweils auf
den Vorgänger, von dem aus ein Zustand zuerst erreicht wurde.

In [1]:
def lade(dateiname):
    global grid, height, width, start, ziel
    f = open(dateiname)
    grid = f.read().splitlines()
    f.close()
    height = len(grid)
    width = len(grid[0])
    for x in range(height):
        for y in range(width):
            if grid[x][y] == 'S':
                start = (x, y)
            elif grid[x][y] == 'Z':
                ziel = (x, y)

lade('maze1.txt')
for zeile in grid:
    print(zeile)
print(f'Höhe = {height}, Breite = {width}')
print(f'{start=}, {ziel=}')

######
#    #
#  Z #
# ####
# S  #
#    #
######
Höhe = 7, Breite = 6
start=(4, 2), ziel=(2, 3)


Für das Labyrinth schreiben wir den Zieltest und die Folgezustände. Die Reihenfolge der Richtungen in `dirs` ist
oben, unten, links, rechts.

In [2]:
dirs = [(-1, 0), (1, 0), (0, -1), (0, 1)]

def goaltest(state):
    return state == ziel

def nextstates(state):
    x, y = state
    tmp = []
    for xd, yd in dirs:
        x1 = x + xd
        y1 = y + yd
        if 0 <= x1 < height and 0 <= y1 < width and grid[x1][y1] != '#':
            tmp.append((x1, y1))
    return tmp

print(nextstates(start))

[(5, 2), (4, 1), (4, 3)]


#### Der Algorithmus

Die Breitensuche verwaltet zwei Dinge:

- **frontier**: eine Warteschlange mit den Zuständen, die schon gefunden, aber noch nicht untersucht wurden. Wir nehmen
  dafür eine `deque` aus dem Modul `collections`. Mit `append` kommt ein Zustand hinten dazu, mit `popleft` nimmt man
  den vordersten heraus. Beides geht schnell, bei einer Liste wäre `pop(0)` langsam.
- **prev**: ein Dictionary, das zu jedem gefundenen Zustand seinen Vorgänger speichert. Der Startzustand hat den
  Vorgänger `None`. Mit `prev` merken wir uns gleichzeitig, welche Zustände wir schon kennen. Jeder Zustand kommt so
  nur einmal in die frontier.

Der Ablauf: Nimm den vordersten Zustand aus der frontier. Ist er das Ziel, sind wir fertig. Sonst kommen alle seine
Folgezustände, die noch nicht in `prev` stehen, hinten in die frontier.

In [3]:
from collections import deque

def bfs(startstate):
    frontier = deque([startstate])
    prev = {startstate: None}
    while frontier:
        state = frontier.popleft()
        if goaltest(state):
            return prev, state
        for v in nextstates(state):
            if v not in prev:
                frontier.append(v)
                prev[v] = state
    return None, None

Am Ende gehen wir vom Ziel aus über `prev` rückwärts bis zum Start und drehen die Liste um. `getMove` beschreibt
einen Zug zwischen zwei aufeinanderfolgenden Zuständen, `getMoves` den ganzen Weg.

In [4]:
def reconstructPath(prev, goalstate):
    state = goalstate
    path = []
    while state is not None:
        path.append(state)
        state = prev[state]
    path.reverse()
    return path

def getMoves(path):
    moves = ''
    for i in range(len(path) - 1):
        moves += getMove(path[i], path[i + 1])
    return moves

def getMove(s1, s2):
    x1, y1 = s1
    x2, y2 = s2
    if x1 < x2: return 'D'   # down
    if x1 > x2: return 'U'   # up
    if y1 < y2: return 'R'   # right
    if y1 > y2: return 'L'   # left

In [5]:
prev, goal = bfs(start)
path = reconstructPath(prev, goal)
print(path)
print(getMoves(path))

[(4, 2), (4, 1), (3, 1), (2, 1), (2, 2), (2, 3)]
LUURR


#### Den Ablauf verfolgen

Wir lassen die Breitensuche noch einmal laufen und geben dabei aus, welcher Zustand untersucht wird und wie die
frontier danach aussieht. Das entspricht dem Baum im Bild oben, Ebene für Ebene von oben nach unten.

In [6]:
def bfs_ablauf(startstate):
    frontier = deque([startstate])
    prev = {startstate: None}
    while frontier:
        state = frontier.popleft()
        if goaltest(state):
            print('Ziel', state)
            return
        for v in nextstates(state):
            if v not in prev:
                frontier.append(v)
                prev[v] = state
        print('untersucht', state, ' frontier:', list(frontier))

bfs_ablauf(start)

untersucht (4, 2)  frontier: [(5, 2), (4, 1), (4, 3)]
untersucht (5, 2)  frontier: [(4, 1), (4, 3), (5, 1), (5, 3)]
untersucht (4, 1)  frontier: [(4, 3), (5, 1), (5, 3), (3, 1)]
untersucht (4, 3)  frontier: [(5, 1), (5, 3), (3, 1), (4, 4)]
untersucht (5, 1)  frontier: [(5, 3), (3, 1), (4, 4)]
untersucht (5, 3)  frontier: [(3, 1), (4, 4), (5, 4)]
untersucht (3, 1)  frontier: [(4, 4), (5, 4), (2, 1)]
untersucht (4, 4)  frontier: [(5, 4), (2, 1)]
untersucht (5, 4)  frontier: [(2, 1)]
untersucht (2, 1)  frontier: [(1, 1), (2, 2)]
untersucht (1, 1)  frontier: [(2, 2), (1, 2)]
untersucht (2, 2)  frontier: [(1, 2), (2, 3)]
untersucht (1, 2)  frontier: [(2, 3), (1, 3)]
Ziel (2, 3)


**Warum findet die Breitensuche den kürzesten Weg?** Die frontier ist eine Warteschlange. Deshalb werden zuerst alle
Zustände untersucht, die einen Zug vom Start entfernt sind, dann alle mit zwei Zügen und so weiter. Der erste Weg zum
Ziel, den sie findet, hat also die kleinste Anzahl von Zügen.

Gibt es mehrere kürzeste Wege, entscheidet die Reihenfolge in `dirs`, welcher gefunden wird.

Die Ebenen kann man sichtbar machen. Wir untersuchen hier alle Felder und schreiben an jedes die Anzahl der Züge
vom Start aus:

In [7]:
def abstaende(startstate):
    frontier = deque([startstate])
    abstand = {startstate: 0}
    while frontier:
        state = frontier.popleft()
        for v in nextstates(state):
            if v not in abstand:
                frontier.append(v)
                abstand[v] = abstand[state] + 1
    return abstand

abstand = abstaende(start)
for x in range(height):
    zeile = ''
    for y in range(width):
        zeile += str(abstand[(x, y)]) if (x, y) in abstand else grid[x][y]
    print(zeile)

######
#4567#
#3456#
#2####
#1012#
#2123#
######


Das Ziel auf Feld $(2, 3)$ hat den Abstand 5, so viele Züge hat auch der Weg `LUURR` oben.

#### Tiefensuche

Nimmt man für die frontier statt einer Warteschlange einen **Stack** (Stapel), entsteht die **Tiefensuche** (dfs,
depth first search). Beim Stack wird der zuletzt abgelegte Zustand als erster wieder herausgenommen. In Python nehmen
wir eine Liste mit `append` und `pop`. Die Tiefensuche läuft so weit wie möglich in eine Richtung und geht erst
zurück, wenn es dort nicht weitergeht.

Weil der zuletzt abgelegte Zustand zuerst drankommt, legen wir die Folgezustände in umgekehrter Reihenfolge ab. So
wird wie bei der Breitensuche zuerst oben, dann unten, links und rechts probiert.

In [8]:
def dfs(startstate):
    frontier = [startstate]              # eine Liste als Stack
    prev = {startstate: None}
    while frontier:
        state = frontier.pop()
        if goaltest(state):
            return prev, state
        folge = nextstates(state)
        folge.reverse()                  # oben wird zuletzt abgelegt und damit zuerst genommen
        for v in folge:
            if v not in prev:
                frontier.append(v)
                prev[v] = state
    return None, None

def zeige(path):
    bild = [list(zeile) for zeile in grid]
    for x, y in path[1:-1]:
        bild[x][y] = 'o'
    for zeile in bild:
        print(''.join(zeile))
    print('Weglänge:', len(path) - 1)

prev, goal = dfs(start)
zeige(reconstructPath(prev, goal))

######
#ooo #
#o Z #
#o####
#oS  #
#    #
######
Weglänge: 7


Die Tiefensuche findet hier einen Weg mit 7 statt 5 Zügen. In einem offenen Labyrinth sieht man den Unterschied
deutlicher:

In [9]:
lade('maze2.txt')
prev, goal = bfs(start)
zeige(reconstructPath(prev, goal))
print('gefundene Zustände:', len(prev))
print()
prev, goal = dfs(start)
zeige(reconstructPath(prev, goal))
print('gefundene Zustände:', len(prev))

############
#          #
#    ##    #
#S   ##oooZ#
#ooooooo   #
############
Weglänge: 11
gefundene Zustände: 36

############
#ooo   ooo #
#o o ##o o #
#S o ##o oZ#
#  ooooo   #
############
Weglänge: 19
gefundene Zustände: 36


Die Tiefensuche geht zuerst so weit nach oben wie möglich und schlängelt sich dann auf und ab zum Ziel.
Sie findet einen Weg mit 19 Zügen, die Breitensuche einen mit 11. Die Tiefensuche findet also einen Weg, wenn es einen
gibt, aber meist nicht den kürzesten.

In der Einheit Graphen 2 wird die Tiefensuche rekursiv geschrieben. Dort wird ein Knoten erst markiert, wenn er besucht
wird, hier schon beim Ablegen in der frontier. Deshalb kann die Reihenfolge der untersuchten Zustände etwas anders
sein. Die Aufgaben zum Labyrinth beziehen sich auf die Fassung mit dem Stack hier.

#### Das BFS-Muster

Die Funktionen `bfs`, `reconstructPath` und `getMoves` können wir für jedes Suchproblem unverändert übernehmen. In
jedem Anwendungsfall müssen wir

- einen Zustand modellieren, meist als Tupel oder String,
- die Funktionen `goaltest` und `nextstates` schreiben,
- falls wir die Züge beschreiben wollen, die Funktion `getMove` schreiben.

Wichtig: Zustände müssen als Schlüssel in `prev` stehen können. Listen gehen deshalb nicht, Tupel und Strings schon.

```python
def nextstates(state):
    '''returns: Liste mit den möglichen Folgezuständen von state'''

def goaltest(state):
    '''returns: True, wenn state eine Lösung ist'''

def getMove(s1, s2):
    '''returns: Beschreibung des Zugs von s1 nach s2'''

# Aufruf:
prev, goal = bfs(startstate)
path = reconstructPath(prev, goal)
print(getMoves(path))
```

`bfs` ruft immer die Funktionen `goaltest` und `nextstates` auf, die gerade definiert sind. In den folgenden
Beispielen definieren wir sie jeweils neu.

#### Beispiel: MOO

Gegeben ist ein String, der nur aus den Zeichen M und O besteht. Folgende Operationen sind möglich:

- ersetze den ersten oder letzten Buchstaben durch den jeweils anderen ('M' wird zu 'O' bzw. 'O' wird zu 'M'),
- lösche den ersten oder letzten Buchstaben.

Ziel ist es, mit möglichst wenigen Operationen aus dem Anfangsstring den String 'MOO' zu erzeugen. Ermittle die Anzahl
der Operationen und beschreibe einen Weg dorthin.

Hinweis: Die Breitensuche ist hier nicht die schnellste Methode, aber zum Üben eignet sich die Aufgabe gut.

    Beispiel: s = 'MOMMOM'

    Lösung in 4 Schritten:
    ändere letzten Buchstaben: MOMMOO
    erster Buchstabe weg: OMMOO
    erster Buchstabe weg: MMOO
    erster Buchstabe weg: MOO

Wir nutzen den String als Zustand.

<img src='moo1.png' width='701'>

Ein String wird nie länger. Strings mit weniger als drei Zeichen können also nie zu 'MOO' werden. Deshalb löschen
wir nur, solange der String länger als 3 ist. Ohne diese Bedingung würde die Suche bei Startstrings ohne Lösung bis
zum leeren String weiterlaufen und dort bei `state[0]` abbrechen.

In [10]:
def nextstates(state):
    tmp = []
    if state[0] == 'M':
        tmp.append('O' + state[1:])
    else:
        tmp.append('M' + state[1:])
    if state[-1] == 'O':
        tmp.append(state[:-1] + 'M')
    else:
        tmp.append(state[:-1] + 'O')
    if len(state) > 3:
        tmp.append(state[1:])
        tmp.append(state[:-1])
    return tmp

def goaltest(state):
    return state == 'MOO'

def getMove(s1, s2):
    if len(s1) > len(s2):
        if s1[1:] == s2:
            return 'erster Buchstabe weg\n'
        return 'letzter Buchstabe weg\n'
    if s1[0] == s2[0]:
        return 'ändere letzten Buchstaben\n'
    return 'ändere ersten Buchstaben\n'

print(nextstates('MOMMOM'))

['OOMMOM', 'MOMMOO', 'OMMOM', 'MOMMO']


In [11]:
for start in ['MOMMOM', 'MMOMOMOMM', 'MMM']:
    prev, goal = bfs(start)
    if prev is None:
        print(start, ': keine Lösung')
    else:
        path = reconstructPath(prev, goal)
        print(start, ':', len(path) - 1, 'Operationen')
        print(getMoves(path))

MOMMOM : 4 Operationen
ändere letzten Buchstaben
erster Buchstabe weg
erster Buchstabe weg
erster Buchstabe weg

MMOMOMOMM : 7 Operationen
erster Buchstabe weg
erster Buchstabe weg
erster Buchstabe weg
erster Buchstabe weg
erster Buchstabe weg
letzter Buchstabe weg
ändere letzten Buchstaben

MMM : keine Lösung


Bei 'MMM' ist der mittlere Buchstabe ein M. Ändern lassen sich nur der erste und der letzte Buchstabe, und löschen
dürfen wir nicht mehr. Die Breitensuche untersucht alle erreichbaren Zustände, findet das Ziel nicht und gibt
`None, None` zurück.

#### Beispiel: Girlande

Eine Girlande besteht aus vier Lampen, die vier verschiedene Farben haben können. Ein String der Länge 4, der nur die
Zeichen 1 bis 4 enthält, stellt eine Girlande dar.

```
girlande = '1211'     # eine Lampe mit Farbe 1, eine mit Farbe 2, dann wieder zwei mit Farbe 1
```

Zu Beginn sind alle Lampen ausgeschaltet. Wir können beliebig oft folgende Operation durchführen: Wähle eine Lampe
und ändere ihren Status (aus *an* mache *aus* und umgekehrt). Die nächste Operation darf aber nur an einer Lampe
ausgeführt werden, die eine andere Farbe hat.

Berechne die minimale Anzahl der Operationen, die wir benötigen, um alle Lampen der Girlande anzuschalten, und
beschreibe eine mögliche Lösung. Gib 'Keine Lösung' aus, wenn es nicht geht.

```
girlande = '1211'
Anzahl Operationen: 6
Lampe 1 ein
Lampe 2 ein
Lampe 3 ein
Lampe 2 aus
Lampe 4 ein
Lampe 2 ein
```

Wie sollen wir einen Zustand modellieren? Jede Lampe wird durch ihre Farbe dargestellt, eine ausgeschaltete Lampe
durch die negative Zahl.

<img src='girlande1.png' width='700'>

Um aus einem Zustand die Folgezustände zu ermitteln, brauchen wir noch die Information, welche Farbe zuletzt
gewechselt wurde. Ein Zustand ist deshalb ein Paar aus den Lampen und der zuletzt benutzten Farbe.

<img src='girlande2.png' width='700'>

In [12]:
def nextstates(state):
    tmp = []
    tup, letzte_farbe = state
    for i in range(4):
        farbe = abs(tup[i])
        if farbe != letzte_farbe:
            b = list(tup)
            b[i] = -b[i]
            tmp.append((tuple(b), farbe))
    return tmp

def goaltest(state):
    return state[0] == ziel

def getMove(s1, s2):
    t1 = s1[0]
    t2 = s2[0]
    for i in range(len(t1)):
        if t1[i] != t2[i]:
            break
    if t2[i] < 0:
        return f'Lampe {i+1} aus\n'
    return f'Lampe {i+1} ein\n'

In [13]:
for girlande in ['1211', '1231', '1111']:
    start = tuple([-int(x) for x in girlande])   # zu Beginn sind alle Lampen ausgeschaltet
    ziel = tuple([int(x) for x in girlande])     # am Ende sind alle Lampen eingeschaltet
    startstate = (start, 0)                       # zu Beginn wurde noch keine Farbe benutzt
    prev, state = bfs(startstate)
    print(f'{girlande=}')
    if prev is None:
        print('Keine Lösung')
    else:
        path = reconstructPath(prev, state)
        print(f'Anzahl Operationen: {len(path)-1}')
        print(getMoves(path))

girlande='1211'
Anzahl Operationen: 6
Lampe 1 ein
Lampe 2 ein
Lampe 3 ein
Lampe 2 aus
Lampe 4 ein
Lampe 2 ein

girlande='1231'
Anzahl Operationen: 4
Lampe 1 ein
Lampe 2 ein
Lampe 3 ein
Lampe 4 ein

girlande='1111'
Keine Lösung


Bei '1111' haben alle Lampen dieselbe Farbe. Nach der ersten Operation ist keine weitere mehr erlaubt.

#### Beispiel: 8-Puzzle

<img src='such_bild1.png' width='500'>

Wir modellieren einen Zustand als Tupel mit der Reihenfolge der Zahlen, die Leerstelle bekommt die 0. Die Züge
beschreiben wir als Bewegung der Leerstelle: U (up), D (down), L (left), R (right).

<img src='8puzzle.png' width='900'>

Die Leerstelle steht im Tupel an Index $i$. Nach oben und unten ändert sich der Index um 3, nach links und rechts
um 1. Ein Nachbar mit Index $j$ muss in derselben Spalte (`i % 3 == j % 3`) oder in derselben Zeile
(`i // 3 == j // 3`) liegen. Sonst würde z.B. die Leerstelle von Index 3 (Mitte links) mit $j = 2$ nach oben rechts
springen.

In [14]:
def goaltest(state):
    return state == (0, 1, 2, 3, 4, 5, 6, 7, 8)

def nextstates(state):
    tmp = []
    i = state.index(0)
    for d in [-3, 3, -1, 1]:
        j = i + d
        if 0 <= j < 9 and (i % 3 == j % 3 or i // 3 == j // 3):
            a = list(state)
            a[i], a[j] = a[j], a[i]
            tmp.append(tuple(a))
    return tmp

def getMove(s1, s2):
    i1 = s1.index(0)   # Position der 0 in s1
    i2 = s2.index(0)   # Position der 0 in s2
    if i2 == i1 - 1: return 'L'
    if i2 == i1 + 1: return 'R'
    if i2 == i1 + 3: return 'D'
    if i2 == i1 - 3: return 'U'

print(nextstates((7, 2, 4, 5, 0, 6, 8, 3, 1)))

[(7, 0, 4, 5, 2, 6, 8, 3, 1), (7, 2, 4, 5, 3, 6, 8, 0, 1), (7, 2, 4, 0, 5, 6, 8, 3, 1), (7, 2, 4, 5, 6, 0, 8, 3, 1)]


In [15]:
startstate = (7, 2, 4, 5, 0, 6, 8, 3, 1)
prev, goal = bfs(startstate)
path = reconstructPath(prev, goal)
print(f'Anzahl Operationen: {len(path)-1}')
print(getMoves(path))
print('Anzahl gefundener Zustände:', len(prev))

Anzahl Operationen: 26
LURDDLURRULLDRRDLURULDDLUU
Anzahl gefundener Zustände: 177809


#### Grenzen der Breitensuche

Beim 8-Puzzle hat die Breitensuche über 170.000 Zustände gefunden, bevor sie das Ziel erreicht hat. Insgesamt kann man
vom Start aus $9!/2 = 181440$ Zustände erreichen. Die Breitensuche hat also fast alle untersucht.

Hat jeder Zustand ungefähr $b$ Folgezustände und liegt das Ziel $d$ Züge entfernt, dann muss die Breitensuche bis zu
$b^d$ Zustände speichern. Beim 15-Puzzle mit $4 \times 4$ Feldern ist das nicht mehr machbar. Dann hilft eine
**Heuristik**, die schätzt, wie weit ein Zustand noch vom Ziel entfernt ist. Damit untersucht man vielversprechende
Zustände zuerst (A*-Suche, Beamsearch).

#### Zum Weiterüben

Bundeswettbewerb Informatik, Runde 1, aus dem Wettbewerbsarchiv:

- Schiebeparkplatz (40. BwInf, Aufgabe 1), dazu gibt es ein [Video](https://youtu.be/MUOExC8rFwA)
- Autoscrabble (36. BwInf, Aufgabe 4)

Codingame:

- [The lost child](https://www.codingame.com/training/medium/the-lost-child-episode-1)
- [Knights Jam](https://www.codingame.com/training/medium/knights-jam)
- [Moves in a maze](https://www.codingame.com/training/easy/moves-in-maze)
- [Six degrees of Kevin Bacon](https://www.codingame.com/training/easy/six-degrees-of-kevin-bacon)
- [2.5D Maze](https://www.codingame.com/training/medium/2-5d-maze)
- [Flood Fill Example](https://www.codingame.com/training/medium/flood-fill-example)
- [Detective Pikaptcha EP3](https://www.codingame.com/training/medium/detective-pikaptcha-ep3)
- [Paper Labyrinth](https://www.codingame.com/training/medium/paper-labyrinth)
- [abcdefghijklmnopqrstuvwxyz](https://www.codingame.com/training/easy/abcdefghijklmnopqrstuvwxyz)